# Fake Image & Photo Tampering Detector — Colab

Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

## 1. Get the project code
Run this cell and click **Choose Files** to upload `tamper-detector.zip`. If you've already uploaded it, the cell skips that step.

In [ ]:
import os
from google.colab import files
if not os.path.exists('/content/tamper-detector.zip'):
    files.upload()  # choose tamper-detector.zip
!unzip -q -o /content/tamper-detector.zip -d /content/
%cd /content/tamper-detector
!pip install -q segmentation-models-pytorch albumentations gradio

## 2. Download CASIA v2.0 from Kaggle
Run this cell and upload the `kaggle.json` you got from Kaggle → Settings → API → *Create New Token*.
If the download fails with a 404, the dataset has moved. Search Kaggle for **"CASIA 2.0 Image Tampering Detection Dataset"**, copy the `owner/name` part of its URL into `KAGGLE_SLUG`, and re-run.

In [ ]:
from google.colab import files
import os
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    files.upload()  # choose kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
KAGGLE_SLUG = 'divg07/casia-20-image-tampering-detection-dataset'
!kaggle datasets download -d $KAGGLE_SLUG -p /content/casia --unzip -q
!find /content/casia -maxdepth 3 -type d | head -20

## 3. (Optional) 2-minute pipeline check on synthetic forgeries
Skip this if you just want the real model.

In [ ]:
!python tools/make_synthetic_dataset.py --out /content/synthetic --n 150
!python train.py --data-root /content/synthetic --epochs 3 --img-size 256 --batch-size 16 --out runs/synthetic

## 4. Train on CASIA v2.0
ResNet34 U-Net on RGB+ELA, BCE + Dice. `--img-size 256` is the fast setting (roughly the 15–25 min budget for 20 epochs on a T4); `384` is more accurate but slower. The best checkpoint (by val pixel F1) is saved to `runs/casia_r34/best.pt` with the image-level threshold calibrated on the validation split.

In [ ]:
!python train.py --dataset casia --data-root /content/casia --epochs 20 --img-size 256 --batch-size 16 --workers 2 --out runs/casia_r34

## 5. Evaluate on the held-out test split

In [ ]:
!python evaluate.py --checkpoint runs/casia_r34/best.pt --split runs/casia_r34/split.json
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('runs/casia_r34/eval_*/example_*.jpg'))[:6]:
    display(Image(f))

## 6. Plot training curves

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open('runs/casia_r34/history.json'))
ep = [r['epoch'] for r in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ep, [r['train_loss'] for r in h], label='train'); ax[0].plot(ep, [r['val_loss'] for r in h], label='val'); ax[0].set_title('loss'); ax[0].legend()
for k in ['pixel_f1', 'pixel_iou', 'image_auc']:
    ax[1].plot(ep, [r.get(k) for r in h], label=k)
ax[1].set_title('validation metrics'); ax[1].legend(); plt.show()

## 7. Test your own edited image
Edit a photo in Photoshop / Canva / an AI inpainting tool, upload it, and run:

In [ ]:
up = files.upload()
for name in up:
    !python predict.py --checkpoint runs/casia_r34/best.pt "$name" --out predictions
    display(Image(f'predictions/{os.path.splitext(name)[0]}_result.jpg'))

## 8. Launch the Gradio demo (public link)

In [ ]:
!python app.py --checkpoint runs/casia_r34/best.pt --share